In [2]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('filament-segmentation-2026')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/filament-segmentation-2026


In [2]:
import os, json, time, warnings, cv2, numpy as np, pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from scipy import ndimage as ndi

try:
    import segmentation_models_pytorch as smp
except ImportError:
    os.system("pip install -q segmentation-models-pytorch albumentations")
    import segmentation_models_pytorch as smp

import albumentations as A
from albumentations.pytorch import ToTensorV2
import pycocotools.mask as mask_utils

CFG = dict(seed=42, crop=768, batch_size=6, epochs=50, lr=3e-4,
           backbone='resnet34', encoder_weights='imagenet', num_workers=2,
           device='cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(s):
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG['seed'])
print("Device:", CFG['device'])

Device: cuda


In [3]:
BASE_DIR = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/")
TRAIN_IMG_DIR = BASE_DIR / "train/train_images"
TEST_IMG_DIR  = BASE_DIR / "test/test_images"
ANN_PATH = BASE_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

with open(ANN_PATH) as f:
    ann = json.load(f)

def seg_to_mask(seg, h, w):
    if isinstance(seg, dict) and 'counts' in seg:
        rle = seg
        if isinstance(rle['counts'], list):
            rle = mask_utils.frPyObjects(rle, h, w)
        elif isinstance(rle['counts'], str):
            rle = {'size': rle['size'], 'counts': rle['counts'].encode()}
        return mask_utils.decode(rle).astype(np.uint8)
    if isinstance(seg, dict) and 'segmentation' in seg:
        return seg_to_mask(seg['segmentation'], h, w)
    if isinstance(seg, list) and len(seg) > 0:
        first = seg[0]
        if isinstance(first, (int, float)):
            seg = [seg]; first = seg[0]
        if isinstance(first, list) and len(first) > 0 and isinstance(first[0], (int, float)):
            if len(first) == 2 and all(len(p) == 2 for p in seg):
                m = np.zeros((h, w), np.uint8)
                cv2.fillPoly(m, [np.round(np.array(seg)).astype(np.int32)], 1)
                return m
            rles = mask_utils.frPyObjects([list(map(float, p)) for p in seg], h, w)
            return mask_utils.decode(mask_utils.merge(rles)).astype(np.uint8)
        if isinstance(first, list):
            m = np.zeros((h, w), np.uint8)
            for poly in seg:
                cv2.fillPoly(m, [np.round(np.array(poly)).astype(np.int32)], 1)
            return m
    raise ValueError(f"Unrecognised segmentation format: {str(seg)[:200]}")

def build_index(ann):
    idx = {}
    id2stem = {im['id']: Path(im['file_name']).stem for im in ann['images']}
    for a in ann['annotations']:
        idx.setdefault(id2stem[a['image_id']], []).append(a['segmentation'])
    return idx

ann_index = build_index(ann)
print("images with annotations:", len(ann_index))

LABEL_CACHE = {}
def get_label_map(stem, h, w):
    if stem in LABEL_CACHE: return LABEL_CACHE[stem]
    lab = np.zeros((h, w), np.uint16)
    for i, seg in enumerate(ann_index.get(stem, []), start=1):
        lab[seg_to_mask(seg, h, w) > 0] = i
    LABEL_CACHE[stem] = lab
    return lab

images with annotations: 707


In [4]:
train_files = sorted(p.name for p in TRAIN_IMG_DIR.glob("*.jpeg"))
test_files  = sorted(p.name for p in TEST_IMG_DIR.glob("*.jpeg"))
df_all  = pd.DataFrame({'filename': train_files})
df_test = pd.DataFrame({'filename': test_files})

groups = df_all['filename'].str[:6]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=CFG['seed'])
tr_idx, va_idx = next(gss.split(df_all, groups=groups))
train_df = df_all.iloc[tr_idx].reset_index(drop=True)
val_df   = df_all.iloc[va_idx].reset_index(drop=True)
print(len(train_df), len(val_df), len(df_test))

nonempty = 0
for fn in train_df.filename[:40]:
    im = cv2.imread(str(TRAIN_IMG_DIR / fn)); h, w = im.shape[:2]
    nonempty += get_label_map(Path(fn).stem, h, w).max() > 0
print(f"non-empty masks: {nonempty}/40")
assert nonempty > 20, "Masks are empty, parser broken"

551 156 180


KeyboardInterrupt: 

In [5]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = A.Compose([
    A.RandomRotate90(p=0.5), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.PadIfNeeded(CFG['crop'], CFG['crop'], border_mode=cv2.BORDER_CONSTANT),
    A.RandomCrop(CFG['crop'], CFG['crop']),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5), A.RandomGamma(p=0.3),
    A.GaussianBlur(p=0.15), A.GaussNoise(p=0.15),
    A.Normalize(MEAN, STD), ToTensorV2()])
eval_tf = A.Compose([A.Normalize(MEAN, STD), ToTensorV2()])

class SolarDataset(Dataset):
    def __init__(self, df, img_dir, tf, mode='train'):
        self.df, self.dir, self.tf, self.mode = df, img_dir, tf, mode
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        fn = self.df.filename.iloc[i]
        img = cv2.cvtColor(cv2.imread(str(self.dir / fn)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        if self.mode == 'test':
            return self.tf(image=img)['image'], fn
        mask = (get_label_map(Path(fn).stem, h, w) > 0).astype(np.uint8)
        out = self.tf(image=img, mask=mask)
        return out['image'], out['mask'].unsqueeze(0).float()

train_loader = DataLoader(SolarDataset(train_df, TRAIN_IMG_DIR, train_tf), batch_size=CFG['batch_size'],
                          shuffle=True, num_workers=CFG['num_workers'], drop_last=True)

model = smp.Unet(CFG['backbone'], encoder_weights=CFG['encoder_weights'],
                 in_channels=3, classes=1).to(CFG['device'])

class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss('binary', from_logits=True)
    def forward(self, p, t): return 0.5 * self.bce(p, t) + 0.5 * self.dice(p, t)

criterion = DiceBCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)
scaler = torch.amp.GradScaler(enabled=CFG['device'] == 'cuda')
print("ready")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

ready


In [6]:
def pad32(x):
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    return F.pad(x, (0, pw, 0, ph), mode='reflect'), h, w

@torch.no_grad()
def predict_prob(model, img, tta=True):
    """8-way test-time augmentation -> probability map at original size."""
    model.eval()
    x, h, w = pad32(img.unsqueeze(0).to(CFG['device']))
    ops = [(k, f) for k in range(4) for f in (False, True)] if tta else [(0, False)]
    acc = 0
    with torch.autocast(CFG['device'], enabled=CFG['device'] == 'cuda'):
        for k, f in ops:
            xi = torch.rot90(x, k, (-2, -1))
            if f: xi = xi.flip(-1)
            p = torch.sigmoid(model(xi)).float()
            if f: p = p.flip(-1)
            acc = acc + torch.rot90(p, -k, (-2, -1))
    return (acc / len(ops))[0, 0, :h, :w].cpu().numpy()

def postprocess(prob, thr=0.5, hi=None, close_k=5, min_area=50, min_score=0.0):
    hi = thr if hi is None else hi
    m = (prob > thr).astype(np.uint8)
    if close_k > 1:
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE,
                             cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k)))
    lab, n = ndi.label(m, structure=np.ones((3, 3)))
    if n == 0:
        return np.zeros(prob.shape, np.int32)
    idx   = np.arange(1, n + 1)
    area  = ndi.sum(m, lab, idx)
    pmax  = ndi.maximum(prob, lab, idx)
    pmean = ndi.mean(prob, lab, idx)
    keep  = (area >= min_area) & (pmax >= hi) & (pmean >= min_score)
    new_id = np.zeros(n + 1, np.int32)
    new_id[1:][keep] = np.arange(1, keep.sum() + 1)
    return new_id[lab]

def dice_score(pred_bin, gt_bin, eps=1e-7):
    inter = (pred_bin & gt_bin).sum()
    return (2 * inter + eps) / (pred_bin.sum() + gt_bin.sum() + eps)

def pq_counts(gt, pr, thr=0.5):
    G, P = int(gt.max()) + 1, int(pr.max()) + 1
    joint = np.bincount(gt.ravel().astype(np.int64) * P + pr.ravel(), minlength=G * P).reshape(G, P)
    ga, pa = joint.sum(1), joint.sum(0)
    inter = joint[1:, 1:]
    iou = inter / np.maximum(ga[1:, None] + pa[None, 1:] - inter, 1)
    match = iou > thr
    tp = int(match.sum())
    return float(iou[match].sum()), tp, int((pa[1:] > 0).sum()) - tp, int((ga[1:] > 0).sum()) - tp

val_ds = SolarDataset(val_df, TRAIN_IMG_DIR, eval_tf, mode='test')

def evaluate(model, params=dict(thr=0.5, close_k=5, min_area=50), cached_probs=None, tta=False):
    sq, TP, FP, FN, dices = 0., 0, 0, 0, []
    probs = {}
    for i in range(len(val_ds)):
        img, fn = val_ds[i]
        prob = cached_probs[fn] if cached_probs else predict_prob(model, img, tta)
        probs[fn] = prob
        gt = get_label_map(Path(fn).stem, *prob.shape)
        pr = postprocess(prob, **params)
        dices.append(dice_score(pr > 0, gt > 0))
        a, tp, fp, fn_ = pq_counts(gt.astype(np.int32), pr)
        sq += a; TP += tp; FP += fp; FN += fn_
    pq = sq / max(TP + 0.5 * FP + 0.5 * FN, 1e-9)
    return dict(dice=float(np.mean(dices)), pq=pq, tp=TP, fp=FP, fn=FN), probs
print("helpers ready")

helpers ready


In [ ]:
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.998), use_buffers=True)

best_pq = -1
for epoch in range(CFG['epochs']):
    model.train(); tl = 0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']}", leave=False):
        x, y = x.to(CFG['device']), y.to(CFG['device'])
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(CFG['device'], enabled=CFG['device'] == 'cuda'):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        ema.update_parameters(model)
        tl += loss.item()
    scheduler.step()
    if epoch < 10:
        print(f"Ep {epoch+1:02d} | loss {tl/len(train_loader):.4f}"); continue
    res, _ = evaluate(ema.module)
    print(f"Ep {epoch+1:02d} | loss {tl/len(train_loader):.4f} | val Dice {res['dice']:.4f} "
          f"| PQ {res['pq']:.4f} | TP {res['tp']} FP {res['fp']} FN {res['fn']}")
    if res['pq'] > best_pq:
        best_pq = res['pq']
        torch.save(ema.module.state_dict(), "best_unet_model.pth")
        print("   saved best")

In [7]:
model.load_state_dict(torch.load("/kaggle/input/models/omnhinge/solar-filament-model-test/pytorch/default/1/best_unet_model (1).pth")); model.eval()

base = dict(thr=0.5, hi=0.5, close_k=5, min_area=50, min_score=0.0)
_, cached = evaluate(model, base, tta=True)

grid = dict(thr=[0.35, 0.45, 0.55],
            hi=[0.5, 0.7, 0.85],
            close_k=[1, 5, 9, 15],
            min_area=[50, 150, 400],
            min_score=[0.0, 0.6])

best_params = dict(base)
best_res, _ = evaluate(model, best_params, cached_probs=cached)
print("start:", best_res)
for rnd in range(2):
    for key, vals in grid.items():
        for v in vals:
            p = {**best_params, key: v}
            r, _ = evaluate(model, p, cached_probs=cached)
            if r['pq'] > best_res['pq']:
                best_params, best_res = p, r
                print(f"round {rnd} {key}={v} -> PQ {r['pq']:.4f} Dice {r['dice']:.4f} "
                      f"TP {r['tp']} FP {r['fp']} FN {r['fn']}")
BEST_PARAMS = best_params
print("Best params:", BEST_PARAMS); print("Val metrics:", best_res)

start: {'dice': 0.6723598544459329, 'pq': 0.29530196704865413, 'tp': 755, 'fp': 847, 'fn': 1016}
round 0 thr=0.55 -> PQ 0.2978 Dice 0.6721 TP 756 FP 827 FN 1015
round 0 hi=0.7 -> PQ 0.3011 Dice 0.6718 TP 755 FP 788 FN 1016
round 0 hi=0.85 -> PQ 0.3113 Dice 0.6712 TP 751 FP 667 FN 1020
round 0 close_k=1 -> PQ 0.3128 Dice 0.6713 TP 759 FP 674 FN 1012
round 0 close_k=15 -> PQ 0.3139 Dice 0.6704 TP 747 FP 613 FN 1024
round 0 min_area=150 -> PQ 0.3184 Dice 0.6703 TP 747 FP 568 FN 1024
round 1 close_k=1 -> PQ 0.3186 Dice 0.6713 TP 759 FP 616 FN 1012
round 1 min_area=400 -> PQ 0.3188 Dice 0.6634 TP 703 FP 440 FN 1068
Best params: {'thr': 0.55, 'hi': 0.85, 'close_k': 1, 'min_area': 400, 'min_score': 0.0}
Val metrics: {'dice': 0.6634389262753958, 'pq': 0.3187877048319305, 'tp': 703, 'fp': 440, 'fn': 1068}


In [8]:
def mask_to_coco_rle(m):
    return mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))['counts'].decode('utf-8')

test_ds = SolarDataset(df_test, TEST_IMG_DIR, eval_tf, mode='test')
RELAXED = dict(thr=0.3, hi=0.3, close_k=5, min_area=30, min_score=0.0)
rows, empty_imgs, t0 = [], 0, time.time()

for i in tqdm(range(len(test_ds))):
    img, fn = test_ds[i]
    prob = predict_prob(model, img, tta=True)
    inst = postprocess(prob, **BEST_PARAMS)
    if inst.max() == 0:
        inst = postprocess(prob, **RELAXED)
    stem = Path(fn).stem
    ids = [k for k in np.unique(inst) if k > 0]
    for j, k in enumerate(ids, 1):
        rows.append(dict(filament_id=f"{stem}_{j}", segmentation_rle=mask_to_coco_rle(inst == k)))
    if not ids:
        empty_imgs += 1
        rows.append(dict(filament_id=f"{stem}_1",
                         segmentation_rle=mask_to_coco_rle(np.zeros(prob.shape, np.uint8))))

sub = pd.DataFrame(rows)
sub.to_csv("submission.csv", index=False)

assert sub.filament_id.is_unique
assert sub.isnull().sum().sum() == 0
assert sub.filament_id.str.rsplit('_', n=1).str[0].nunique() == len(df_test), "missing test images"
assert not sub.segmentation_rle.str.contains('"').any()
print(f"Inference {time.time()-t0:.0f}s | rows {len(sub)} | empty images {empty_imgs} | "
      f"avg filaments/img {len(sub)/len(df_test):.1f}")
print(sub.head())

100%|██████████| 180/180 [05:05<00:00,  1.70s/it]

Inference 305s | rows 1341 | empty images 2 | avg filaments/img 7.5
          filament_id                                   segmentation_rle
0  20110120105534Ch_1  m_Zi24lo12O1N2hPNJcn16[QNLen14ZQNMfn14YQNKin17...
1  20110120105534Ch_2  VjRQ24ko12M2O001O0O2O0O10O1O0100O1O10000L4N2M2...
2  20110130110334Ch_1  c_ld22lo18I3M3M7J2OO01QQN[Obn1e0ZQN_Ogn1l02O0O...
3  20110214175414Mh_1  X]b[28eo14N1O2M3I6O2N101POVOgRNk0Wm1YOgRNh0Wm1...
4  20110214175414Mh_2  a]Vb24lo11O1O1O001O0010O010O10O1O0000001OO10O1...
